### Task 1: Load a Pre-trained LLM and Experiment with Prompt Engineering

In [37]:
from transformers import pipeline

# --- Task 1:

generator = pipeline("text-generation", model="gpt2")

# -- initial prompt

# prompts model
prompt = "Explain how to create a Machine Learning Model in simple terms"
# defines starting sentence, max output length, # of answers
output = generator(prompt, max_length=100, num_return_sequences=1,
# controls creativity (0.7 is medium), next word chosen from top 50 words
temperature = 0.7, top_k=50)
print(output[0]['generated_text'])

# -- prompt modification
# Casual Prompt

prompt = "What is a transformer model?"
output = generator(prompt, max_length=100, num_return_sequences=1, temperature = 0.7, top_k=50)
print(output[0]['generated_text'])

# Detailed Prompt

prompt = "Explain how to create a Machine Learning Model in simple terms for a 10 year old child."
output = generator(prompt, max_length=100, num_return_sequences=1, temperature = 0.7, top_k=50)
print(output[0]['generated_text'])

# Structured Prompt

prompt = "Give me a step-by-step guide on how to build a simple transformer model in 2 hours"
output = generator(prompt, max_length=100, num_return_sequences=1, temperature = 0.7, top_k=50)
print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain how to create a Machine Learning Model in simple terms, using the PLEX and RLP libraries.

In short, get a machine learning model that you can run on your local machine and it will compute the results. You can also use this model in your own project to demonstrate your application.

Note: The above example is an example of a basic Machine Learning model and not a complete machine learning model.

You can get a list of all your Machine Learning models in the following format:

rLists (in this case, the list of all your models in the library)

(in this case, the list of all your models in the library) RLP models (in this case, the list of all your models in the library)

(in this case, the list of all your models in the library)

(in this case, the list of all your models in the library)

You can use the below examples to get a list of all your models.

rLists (in this case, the list of all your models in the library)

(in this case, the list of all your models in the library)

(

KeyboardInterrupt: 

### Task 2: Use Hugging Face API or any other closed-source LLM API (e.g., OpenAI GPT, Claude) for Text Generation

In [33]:
# --- Task 2 --> OpenAI
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()  # loads .env variables


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
model="gpt-4o-mini",
messages=[{"role": "user", "content": "Explain Artificial Intelligence in simple terms."}],
)
print(response.choices[0].message.content)

Artificial Intelligence, or AI, is a way for machines, like computers, to mimic human intelligence. This means they can perform tasks that usually require human thinking, such as learning from experience, solving problems, understanding language, and making decisions. 

Think of AI as a smart assistant that can help with things like answering questions, recognizing faces in photos, recommending movies you might like, or even driving a car. It does this by using data and algorithms (a set of rules or instructions) to learn and improve over time, just like how we learn from our experiences.


### Task 3: Instruction Fine-Tune a Tiny Model on a Mini Dataset and Compare Outputs

In [1]:
!pip install -U transformers accelerate trl datasets

In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

# small+fast
MODEL_NAME = "distilgpt2"

# Load dataset
ds = load_dataset("json", data_files="toy_instructions.jsonl")["train"]

# Formatting into a single text field for instruction tuning
def format_example(ex):
    return {
       "text": f"### Instruction:\n{ex['instruction']}\n"
    f"### Input:\n{ex['input']}\n"f"### Response:\n{ex['output']}"
    }

ds = ds.map(format_example)

# Load model + tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Train (kept tiny)
cfg = SFTConfig(
    output_dir="sft_out",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="no",
    eos_token=tok.eos_token,
    pad_token=tok.pad_token,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=cfg,
)

trainer.train()
trainer.save_model("sft_model")
tok.save_pretrained("sft_model")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Step,Training Loss
5,3.836280
10,2.662162
15,2.009030
20,1.581615
25,1.345527
30,1.053076
35,1.107251
40,1.017458
45,0.919491
50,0.813166


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('sft_model/tokenizer_config.json', 'sft_model/tokenizer.json')

In [14]:
# Tests base model and fine-tuned model on 3 prompts and compares results
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n###Input:\n{inp}\n ### Response:\n"

def make_gen(model_name_or_path: str):
    tok = AutoTokenizer.from_pretrained(model_name_or_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
    gen = pipeline("text-generation", model=model, tokenizer=tok)
    return gen

def run_one(gen, instruction, inp, max_new_tokens=60):
    prompt = PROMPT_TEMPLATE.format(instruction=instruction, inp=inp)
    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
    )[0]["generated_text"]
    return out[len(prompt):].strip()

tests = [
    ("Rewrite the sentence to be more polite.", "Give me that."),
    ("Classify the sentiment as positive/negative.", "This is the worst."),
    ("Extract the city from the text.", "I drove from LA to San Diego."),
]

# BEFORE (base)
base_gen = make_gen("distilgpt2")

# AFTER (fine-tuned)
tuned_gen = make_gen("sft_model")

for inst, inp in tests:
    print("IN:", (inst, inp))
    print("BEFORE:", run_one(base_gen, inst, inp))
    print("AFTER :", run_one(tuned_gen, inst, inp))
    print("-" * 60)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

IN: ('Read the sentence and then answer the question.', ' John went to Paris in 2019 and moved to Berlin in 2021. Where does John live now?')


KeyboardInterrupt: 

### Part C -- Findings and Reflection

In [11]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

# small+fast
MODEL_NAME = "distilgpt2"

# Load dataset
ds = load_dataset("json", data_files="toy_instructions_2.jsonl")["train"]

# Formatting into a single text field for instruction tuning
def format_example(ex):
    return {
       "text": f"### Instruction:\n{ex['instruction']}\n"
    f"### Input:\n{ex['input']}\n"f"### Response:\n{ex['output']}"
    }

ds = ds.map(format_example)

# Load model + tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Train (kept tiny)
cfg = SFTConfig(
    output_dir="sft_out",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="no",
    eos_token=tok.eos_token,
    pad_token=tok.pad_token,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=cfg,
)

trainer.train()
trainer.save_model("sft_2_model")
tok.save_pretrained("sft_2_model")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss
5,3.523381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('sft_2_model/tokenizer_config.json', 'sft_2_model/tokenizer.json')

In [15]:
# Tests base model and fine-tuned model on 3 prompts and compares results
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n###Input:\n{inp}\n ### Response:\n"

def make_gen(model_name_or_path: str):
    tok = AutoTokenizer.from_pretrained(model_name_or_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
    gen = pipeline("text-generation", model=model, tokenizer=tok)
    return gen

def run_one(gen, instruction, inp, max_new_tokens=50):
    prompt = PROMPT_TEMPLATE.format(instruction=instruction, inp=inp)
    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
    )[0]["generated_text"]
    return out[len(prompt):].strip()

tests = [
("Read the sentence and then answer the question.", " John went to Paris in 2019 and moved to Berlin in 2021. Where does John live now?"),
]

# BEFORE (base)
base_gen = make_gen("distilgpt2")

# AFTER (fine-tuned)
tuned_gen = make_gen("sft_2_model")

for inst, inp in tests:
    print("IN:", (inst, inp))
    print("BEFORE:", run_one(base_gen, inst, inp))
    print("AFTER :", run_one(tuned_gen, inst, inp))
    print("-" * 60)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

IN: ('Read the sentence and then answer the question.', ' John went to Paris in 2019 and moved to Berlin in 2021. Where does John live now?')


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE: John is a freelance writer, journalist, author of The New York Times bestseller "The Life of Pablo," as well as an avid reader of books on politics, history & culture. He has written for several magazines including American Review (New Yorker),
AFTER : !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
------------------------------------------------------------
